# 🧪 W4-D5 概念实验：chunk 大小、overlap、top-K 三个旋钮怎么拧？

> 配套阅读：`ima/第4周-Day5-RAG实战架构设计与优化.md`（生产级管道、分块策略对比表、优化速查表在那边）
>
> Demo 级 RAG 上不了线，问题往往出在三个"看起来人畜无害"的参数上。这个 notebook 用模拟实验回答：
> 1. **chunk_size**：切太小事实被切断；切太大检索照样命中，但"能答对"反而下降——为什么？
> 2. **overlap**：多花 40% 索引体积，到底买回了什么？
> 3. **top-K**：K 越大召回越高，为什么答案质量先升后降？（lost in the middle 模拟）
>
> 实验环境：纯 numpy 模拟检索。为消除切窗相位带来的随机性，每个宽度扫 20 个偏移取平均。

## 实验 1：chunk 宽度扫描 —— 分两个口径看

构造"每句一个产品事实（名字+价格）+ 填充句"的文档，固定宽度滑窗切割，bigram 余弦检索。
判分分两个口径，这是本实验的关键设计：

- **检索完整性**：Top-1 chunk 是否同时包含产品名和价格（检索阶段的 recall）；
- **可答性**：检索完整性 ∧ chunk 里**其他产品的价格 ≤ 1 个**（模拟生成阶段：chunk 混入多个价格时 LLM 容易串行——对应 md 里"chunk 太大稀释信号"的误区）。

In [ ]:
import re
import numpy as np

rng = np.random.default_rng(42)

PRODUCTS = [
    ("杨枝甘露", "泰国椰浆", 22), ("芒果双皮奶", "顺德牛奶", 18), ("芋泥波波冰", "荔浦芋头", 17),
    ("西瓜冰", "麒麟西瓜", 16), ("双皮奶", "水牛奶", 15), ("红豆沙", "云南红豆", 12),
    ("桂花酸梅汤", "乌梅山楂", 10), ("椰汁西米露", "海南椰汁", 14),
]
FILLERS = ["营业时间为早十点到晚十点。", "扫码点单可享会员积分。", "支持预约外送服务。", "节假日正常营业欢迎光临。"]
facts = [f"{n}选用{m}制作，售价{p}元。" for n, m, p in PRODUCTS]
BASE_DOC = "".join(f + FILLERS[i % 4] for i, f in enumerate(facts))
LEAD = "欢迎光临糖水小铺，本段为排版填充文字。"   # 用不同截断长度制造切窗相位偏移

def bigrams(t):
    t = re.sub(r"[，。、；：\s]", "", t)
    return [t[i:i+2] for i in range(len(t) - 1)]

def fact_spans(d):
    return [(d.index(f"{name}选用"), len(f"{name}选用{m}制作，售价{price}元。"), name, price)
            for name, m, price in PRODUCTS]

def eval_width(width, overlap, n_offset=20):
    step = max(1, round(width * (1 - overlap)))
    surv, ok_hits, n_chunks = [], [], []
    for off in range(n_offset):
        d = LEAD[off % len(LEAD):] + BASE_DOC
        chunks = [d[i:i+width] for i in range(0, max(1, len(d) - width // 2), step)]
        starts = list(range(0, max(1, len(d) - width // 2), step))
        # 口径1：事实存活率 = 完整事实落在至少一个 chunk 里（纯切割几何，与检索器无关）
        sv = 0
        for pos, flen, _, _ in fact_spans(d):
            if any(st <= pos and pos + flen <= st + width for st in starts):
                sv += 1
        surv.append(sv / len(PRODUCTS))
        # 口径2：可答率 = Top-1 chunk 含完整事实 且 其他产品价格 ≤1（端到端，含生成串扰模拟）
        vocab = sorted({g for c in chunks for g in bigrams(c)})
        gidx = {g: i for i, g in enumerate(vocab)}
        def vec(t):
            v = np.zeros(len(vocab))
            for g in bigrams(t):
                if g in gidx:
                    v[gidx[g]] += 1
            n = np.linalg.norm(v)
            return v / n if n > 0 else v
        M = np.stack([vec(c) for c in chunks])
        ok = 0
        for name, _, price in PRODUCTS:
            best = chunks[int(np.argmax(M @ vec(f"{name}多少钱")))]
            ret = (name in best) and (f"{price}元" in best)
            other = len(re.findall(r"\d+元", best)) - (1 if ret else 0)
            ok += ret and other <= 1
        ok_hits.append(ok / len(PRODUCTS))
        n_chunks.append(len(chunks))
    return float(np.mean(surv)), float(np.mean(ok_hits)), float(np.mean(n_chunks))

WIDTHS = [12, 16, 20, 24, 28, 36, 48, 64, 96]
SWEEP = {0.0: [eval_width(w, 0.0) for w in WIDTHS],
         0.3: [eval_width(w, 0.3) for w in WIDTHS]}

print(f"{'宽度':>4} | {'事实存活率':>8} {'可答率(无重叠)':>11} {'可答率(30%重叠)':>12}")
for w, (r0, o0, _), (_, o3, _) in zip(WIDTHS, SWEEP[0.0], SWEEP[0.3]):
    print(f"{w:>5} | {r0:>9.0%} {o0:>12.0%} {o3:>14.0%}")
print()
print("→ 事实存活率随宽度单调上升（无重叠时 ≈ (w-事实长+1)/w，永远到不了100%——总有事实跨在边界上）；")
print("→ 可答率倒U形：太小被切断，太大 chunk 混入多个价格 → 生成阶段串扰（96字档崩掉）。")

## 实验 2：宽度曲线可视化 + overlap 到底买回了什么

把实验 1 的两条曲线画出来，再叠加 30% overlap 版的可答性曲线：
overlap 的收益集中在"存活率爬坡段"的临界区（本例 24~36 字）——那里的事实最容易被边界斩断。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

surv0 = [r for r, _, _ in SWEEP[0.0]]
ok0  = [o for _, o, _ in SWEEP[0.0]]
ok3  = [o for _, o, _ in SWEEP[0.3]]

fig, ax = plt.subplots(figsize=(8, 4.4))
ax.plot(WIDTHS, surv0, "o-", color="#4f81bd", label="事实存活率（完整事实在≥1个chunk里）")
ax.plot(WIDTHS, ok0, "s-", color="#c0504d", label="可答性（无重叠）")
ax.plot(WIDTHS, ok3, "^--", color="#9bbb59", label="可答性（30%重叠）")
ax.axvspan(18, 28, alpha=0.08, color="gray")
ax.annotate("≈一条完整事实的长度", (23, 0.05), ha="center", fontsize=9, color="gray")
ax.set_xlabel("chunk 宽度（字符）")
ax.set_ylabel("命中率（20个切窗相位平均）")
ax.set_title("chunk 太小→事实被切断；太大→检索命中但生成串扰（可答性下降）")
ax.legend(loc="center right")
plt.tight_layout()
plt.show()

ramp = [i for i, w in enumerate(WIDTHS) if 24 <= w <= 36]
gain_ramp = np.mean([ok3[i] - ok0[i] for i in ramp])
gain_big = np.mean([ok3[i] - ok0[i] for i in range(len(WIDTHS) - 3, len(WIDTHS))])
print(f"overlap 在存活率爬坡区(24~36字)平均买回可答性：{gain_ramp:+.0%}")
print(f"overlap 在大窗口区(≥48字)平均买回可答性：{gain_big:+.0%}（基本白花钱，只剩索引膨胀）")

## 实验 3：top-K 的甜蜜点 —— 召回越多，答案不一定越好

模拟 40 个 chunk（6 个相关、34 个噪声），检索分有重叠分布。
K 增大：recall 单调上升、precision 下降是常识；关键是**"lost in the middle"**——
LLM 对长上下文中间位置的注意力弱（U 形），且噪声越多串扰越大。两个衰减都乘进去，看有效利用率在哪见顶。

In [ ]:
import numpy as np

rng = np.random.default_rng(7)
N_REL = 6
scores = np.concatenate([rng.normal(0.72, 0.08, N_REL), rng.normal(0.50, 0.13, 40 - N_REL)])
is_rel = np.concatenate([np.ones(N_REL), np.zeros(40 - N_REL)])
order = np.argsort(-scores)
rank_rel = np.where(is_rel[order] == 1)[0] + 1   # 6 个相关 chunk 的最终名次

ks = np.arange(1, 21)
recall = np.array([np.mean(rank_rel <= k) for k in ks])
precision = np.array([np.sum(rank_rel <= k) / k for k in ks])

def effective(k):
    # ① 注意力U形：首位/末位0.95，正中间0.45
    if k == 1:
        w = np.array([0.95])
    else:
        w = 0.45 + 0.5 * np.abs(np.cos(np.linspace(0, np.pi, k)))
    p_seen = 1 - np.prod([1 - w[r - 1] for r in rank_rel if r <= k])  # 至少一个相关chunk被有效注意到
    # ② 噪声串扰：prompt里混入的噪声chunk越多，生成阶段跑偏概率越大（模拟）
    n_noise = k - np.sum(rank_rel <= k)
    focus = np.exp(-0.09 * n_noise)
    return p_seen * focus

eff = np.array([effective(k) for k in ks])
best_k = ks[int(np.argmax(eff))]

print(f"{'K':>3} {'Recall':>8} {'Precision':>10} {'有效利用率':>10}")
for k in [1, 3, 5, 8, 12, 20]:
    i = k - 1
    print(f"{k:>3} {recall[i]:>8.0%} {precision[i]:>10.0%} {eff[i]:>10.0%}")
print(f"\n有效利用率峰值出现在 K={best_k}（recall 还没到 100%，但再堆 chunk 噪声串扰开始反噬）")
print("对应 md 结论：'检索结果越多越好'是误区，K 要对着端到端评测调，不是对着 recall 调。")

## 实验 4：overlap 的成本账

拿实验 1/2 的数据算总账：step 缩短 30% → chunk 数大约 +43%。
看这笔索引开销在每个宽度档位买回了多少可答性。

In [ ]:
import numpy as np   # 复用实验1的 WIDTHS / SWEEP

print(f"{'宽度':>4} {'chunk数(0%)':>10} {'chunk数(30%)':>11} {'索引膨胀':>8} {'可答性提升':>10}")
for w, (_, o0, n0), (_, o3, n3) in zip(WIDTHS, SWEEP[0.0], SWEEP[0.3]):
    print(f"{w:>5} {n0:>10.1f} {n3:>11.1f} {n3/n0-1:>+8.0%} {o3-o0:>+10.0%}")

mean_growth = np.mean([n3 / n0 for (_, _, n0), (_, _, n3) in zip(SWEEP[0.0], SWEEP[0.3])]) - 1
print(f"\n平均索引膨胀 {mean_growth:+.0%}；结论（对照 md 的分块策略表）：")
print("  1) chunk_size 对齐'一条完整事实'的长度（本实验≈20-28字），检索和可答性同时最优；")
print("  2) overlap 只在临界宽度区间有价值，宽 chunk 本身跨多个事实——花钱买不回任何东西；")
print("  3) top-K 对着'有效利用率'调（实验3峰值 K=%d），K 拉满是把噪声塞给 LLM。" % best_k)

## 结论

| 旋钮 | 拧错了会怎样 | 实验证据 |
|---|---|---|
| chunk_size | 太小→事实被切断；太大→检索命中但生成串扰 | 实验1/2：完整性单调升，可答性倒U形 |
| overlap | 边界事实丢失 | 实验2/4：只在临界宽度区买回可答性，索引+43% |
| top-K | 盲目调大→中间遗忘+噪声串扰 | 实验3：有效利用率在中等K见顶后反噬 |

→ 深入阅读：`ima/第4周-Day5-RAG实战架构设计与优化.md`（生产管道五层架构、上下文优化、生成约束、性能速查表）